In [1]:
"""LRが逆になっている場合は、こっちのコードではなく、右タブの1.0.5LRとなっているコードです。
基本的にこのタブでコードを実行すればOK
"""
import openpyxl
import glob
import os
from openpyxl import Workbook
import numpy as np
import pandas as pd
import xlwings as xw
import csv
import re

"""
平均Hzの追加
コードの微修正（エクセルの関数入力の箇所）
サマリーシートを最後に先頭行にする
平均Hzを追加したので、時刻のデータをxlwingsで更新
時刻データがある場合は、fstrで入力できない（なぜかは不明）ので,str＋方式で書き直した
"""

'\n平均Hzの追加\nコードの微修正（エクセルの関数入力の箇所）\nサマリーシートを最後に先頭行にする\n平均Hzを追加したので、時刻のデータをxlwingsで更新\n時刻データがある場合は、fstrで入力できない（なぜかは不明）ので,str＋方式で書き直した\n'

In [6]:
# まとめた後のexcelファイル名
after_name = "gathering data"

# まとめる前のexcelファイルが保存されているパス
folderpath_before = "/Users/shun/Desktop/Python/data"
# まとめた後のexcelファイルを保存するパス
folderpath_after = "/Users/shun/Desktop/Python/summary"




# データを抽出する初めのセル
#iatの指定：行番号は0が始まりのため、エクセル上の座標よりマイナス１すru
# Lのタイトル、値
title = 'B12'
title_row, title_col = 12-1,2-1
val_star, val_col = 18-1, 2

# Rのタイトル、値
tiR = 'C12'
tiR_row, tiR_col = 12-1,3-1

valR,valR_col = 18-1,4



In [7]:
wb = Workbook()

files = sorted(glob.glob(os.path.join(folderpath_before, "*.csv")), key=lambda x: int(re.search(r'\d+', os.path.basename(x)).group()))
new_ws = wb.active




def normalize_csv(filepaths):
    # 全ファイルの処理結果を格納するためのリストを用意する
    rows_list = []
    # すべてのファイルに対して処理を行う
    for filepath in filepaths:
    # CSVファイルを開く
        with open(filepath) as f:
            # CSVファイルを一気に読み込む
            rows = list(csv.reader(f))

            # フィールド数の最大値を取得する
            max_fields = max(len(row) for row in rows)

            # フィールド数が揃うまで空文字を追加する
            for row in rows:
                while len(row) < max_fields:
                    row.append('')
            #空文字の挿入が末尾以外の場合、CSVの仕様上、たとえば先頭行にコンマを入れるケースは存在しない。
                #for i in range(max_fields):
                 #   if i >= len(row):
                  #      row.insert(i, '')

        # 処理結果を格納する
        rows_list.append(rows)

    # 全ファイルの処理結果をまとめたリストを返す
    return rows_list


normalized_rows = normalize_csv(files)



# セルの初期座標を設定
x_one, y_one, y_two = 1, 1, 1

new_ws_titles=[]
flg=True

# Lをまとめる
for file in normalized_rows:

    
    df = pd.DataFrame(file)
    new_ws_titles.append(df.iat[title_row,title_col])
    new_ws.cell(x_one, y_one).value = df.iat[title_row,title_col]
    y_one += 1
    x_two = 2
    for cell in df.loc[val_star:, val_col]:
        new_ws.cell(x_two, y_two).value =  cell
        x_two += 1
    y_two += 1    
    
# Rをまとめる
for file in normalized_rows:
    
    df = pd.DataFrame(file)
    new_ws_titles.append(df.iat[tiR_row,tiR_col])
    new_ws.cell(x_one, y_one).value = df.iat[tiR_row,tiR_col]
    y_one += 1
    x_two = 2
    for cell in df.loc[valR:, valR_col]:
        new_ws.cell(x_two, y_two).value =  cell
        x_two += 1
    y_two += 1

# データが抽出できているか確認    
#print(str(new_ws_titles))

wb.save(folderpath_after + "/" + after_name + ".xlsx")
wb.close()




print("次の処理を行ってください")

次の処理を行ってください


In [8]:
# データ解析のexcelファイル名を決める
save_file_name = "Summary"


# gathering data.xlsxがあるパス名およびデータ解析したファイルを保存するパス名
fileload_path = "/Users/shun/Desktop/Python/summary/"+after_name+".xlsx"
filesave_path = r"/Users/shun/Desktop/Python/summary"


#時刻データ00:00:00 でデータを解析するならTrueに、しないならFalseに
date_flag=False
#その時刻データはどのセルにありますか？　セル座標をｘ，ｙの順で入力
first_time_cell_x, cell_y=18-1,2-1

#集計時間は何秒ですか？
data_interval=1
#Pause_criteria(バーストが終わったとみなす空白の時間)は何秒ですか？
Pause_criterion = 1


#1/data_interval*Pause_criterion:仕様上、データの集計間隔は1secを想定している。
#そのため、定数1を集計間隔data_intervalで割り、その倍率でPause_criterionを補正する。
adjusted_criterion=int(1/data_interval*Pause_criterion)

In [9]:

wb2 = Workbook()
wb2 = openpyxl.load_workbook(fileload_path)
gathering_ws = wb2.worksheets[0]
excel = Workbook()
Sheet_one = excel.active
excel.create_sheet(title='TimeAndIBI')
interval_ws=excel.worksheets[-1]
excel.create_sheet(title='lick_rate')
lick_rate_ws=excel.worksheets[-1]






def get_burst(arr, N):
    """最初に現れた0でない値から、その次の0がN連続するまでの間を取得
    """
    arr = np.r_[arr, np.zeros(N, dtype=int)]#最後にゼロを追加：ゼロがn回続けば一つの塊として処理するため、続かない場合のエラーを回避する
    is_nonzero = arr != 0# 0=F, 0>Tとすることで0>の区別を消す
    # 始点＝最初に現れた0以外の値の位置(リスト内が論理演算のため、TRUEが大きく一番先に検出したTを返す)
    #そのためには、bool型をNumPy配列型に変換する必要がある（EX.　np.array(bool_data)）。
    s = is_nonzero.argmax()
    ln = np.convolve(is_nonzero[s:], np.ones(N, dtype=int), 'valid').argmin()# sより後ろで初めて0がN連続する箇所のindex
    #備考：論理演算子に対してconvolve関数（第２引数が１）でフィルタするとTを１に、Fを０にすることができる

    
    return arr[s:s+ln], (s, s+ln), s, ln#クラスターを返す、クラスターの開始時間マス、クラスターの時間, interBintervl


# y座標を作成
index_two, index_ws_time=1,1



#最大列の値を取得
col_end =gathering_ws.max_column

data,time_list,IBI_list=[],[],[]
lick_rate_list, lick_rate_time =[],[]

"""クラスターを算出する
"""
file_R_index=0
for index, j in enumerate(range(1,col_end+1)):
    now_col_let=openpyxl.utils.get_column_letter(j)
    datas= [cell.value for cell in gathering_ws[now_col_let]]
    datas.pop(0)

    if index >= col_end/2:
        df = pd.DataFrame(normalized_rows[file_R_index])
        file_R_index+=1
    else :
        df = pd.DataFrame(normalized_rows[index])
    
    for dat in datas:
        if dat is None:
            continue
        data.append(dat)
    np_data = np.array(data, dtype='int64')
    j = []
    start_lick=[]
    time_list.clear()
    IBI_list.clear()
    lick_rate_list.clear()
    lick_rate_time.clear()
    pre_burst_end=0
    for index, i in enumerate(np_data):
        if i >=1:
            lick_rate_list.append(i)
            lick_rate_time.append(index)

    for index, v in enumerate(lick_rate_time):#lickの開始時間を書き込む
        if date_flag == True:
            date = str(df.iat[first_time_cell_x + v, cell_y])
            target = ':'
            idx = date.find(target)-2
            r = date[idx:] 
            r2 = r.replace(' ', '')
            lick_rate_ws.cell(index+3, index_two).value = r2
            lick_rate_ws.cell(index+3, index_two).number_format= "h:mm:ss"
        else :
             lick_rate_ws.cell(index+3, index_two).value = v/int(1/data_interval)
            #start_lickはセルのインデックスが収納されているため、補正する
    for index, v in enumerate(lick_rate_list):
        lick_rate_ws.cell(index+3, index_two+1).value = v
    
    
    
    for i in range(np_data.size):
        out, [s, e], st_l, time= get_burst(np_data, adjusted_criterion)
        
        
        j.append(out.sum())#リック内の０も含むがmeanでなくsumなので問題なし
        time_list.append(time)#indexが返されるが、終了時刻は使用上ずれるため、加算しない
        start_lick.append(st_l)#indexが返される
        np_data[:e] = 0
        
        
        if (pre_burst_end > 0) and (out.sum() != 1):
            post_burst_str= st_l+1
            IBI_list.append(post_burst_str-pre_burst_end)
        
        if np_data.sum() == 0:
            break    

        if out.sum() != 1:
            pre_burst_end= st_l+1+time  
        
    if 1 in j:#リックが１回しかないクラスターを削除する
        del_list=[]
        del_list.clear()
        index_num = [n for n, v in enumerate(j) if v == 1]
        for index, v in enumerate(index_num):
            j.pop(v-index)
            start_lick.pop(v-index)
            time_list.pop(v-index)
            
    for index, v in enumerate(start_lick):#クラスターの開始時間を書き込む
        if date_flag == True:
            date = str(df.iat[first_time_cell_x + v, cell_y])
            target = ':'
            idx = date.find(target)-2
            r = date[idx:] 
            r2 = r.replace(' ', '')
            Sheet_one.cell(index+3, index_two).value = r2
            Sheet_one.cell(index+3, index_two).number_format= "h:mm:ss"
        else :
             Sheet_one.cell(index+3, index_two).value = v/int(1/data_interval)
            #start_lickはセルのインデックスが収納されているため、補正する
            
    index_two += 1
    for index, v in enumerate(j):#クラスターサイズを記入
        Sheet_one.cell(index+3, index_two).value = v
    index_two += 1
    for index, v in enumerate(time_list):#なめ時間を書き込む
        interval_ws.cell(index+3, index_ws_time).value = v
    index_ws_time += 1
    for index, v in enumerate(IBI_list):#IBIを書き込む
        interval_ws.cell(index+3, index_ws_time).value = v
    index_ws_time += 1
    
    data.clear()
    

header_list,header_xy=[],[]

header_list.clear
header_xy.clear
for headers in gathering_ws.iter_rows(min_row=1, max_row=1):#headerの取得
    for index, header in enumerate(headers):
        Sheet_one.cell(1,2+index*2).value=header.value
        header_list.append(header.value)

Sheet_one.insert_rows(2,1)#headrを入力
interval_ws.insert_rows(1,1)#のちに入れるAverageIFs関数が値と参照するセルがx軸表（列）が同じでなければいけないため
Sheet_one.cell(2,1).value, Sheet_one.cell(2,2).value="burst start time", "burst size"
interval_ws.cell(2,1).value, interval_ws.cell(2,2).value="licking time", "IBI"







"""新しいシートにデータをまとめる
"""


excel.create_sheet(title='Summary')
summary_ws=excel.worksheets[-1]

#新しいワークシートのheadrの取得
summary_ws.cell(1,2).value="Ave.Burst size(Palatability)"
summary_ws.cell(1,len(set(header_list))+3).value="Number of Burst(Motivation)"    
summary_ws.cell(1,len(set(header_list))*2+4).value="Total Licks(times)"     
summary_ws.cell(1,len(set(header_list))*3+5).value="Licking time(sec/burst)"
summary_ws.cell(1,len(set(header_list))*4+6).value="IBI(sec)"
summary_ws.cell(1,len(set(header_list))*5+7).value="Licking rate(Hz)"


for index, header in enumerate(sorted(set(header_list), key=header_list.index)):
    summary_ws.cell(1,3+index).value=header
    summary_ws.cell(1,4+len(set(header_list))+index).value=header    
    summary_ws.cell(1,5+len(set(header_list))*2+index).value=header
    summary_ws.cell(1,6+len(set(header_list))*3+index).value=header
    summary_ws.cell(1,7+len(set(header_list))*4+index).value=header
    summary_ws.cell(1,8+len(set(header_list))*5+index).value=header

#時刻とラベル,データ処理順を記載
summary_ws.cell(1,1).value="time slot"
if date_flag == True:
    summary_ws.cell(2,1).value="00:00:00"
    summary_ws.cell(3,1).value="23:59:59"
else:
    summary_ws.cell(2,1).value=0
    summary_ws.cell(3,1).value=86401
    
summary_ws.cell(5,1).value="Data processing order"
for index, v in enumerate(files):
    summary_ws.cell(6+index,1).value= os.path.basename(v)

excel.save(filesave_path + "/" + save_file_name + ".xlsx")




'''
各個体のAVE、COUNT、SUMを算出する
'''
max_row = Sheet_one.max_row
max_col = Sheet_one.max_column
max_col2 =lick_rate_ws.max_row
max_row2 =lick_rate_ws.max_column
for cells in Sheet_one.iter_rows(min_row=1, max_row=1):
    for cell in cells:
        if cell.value is not None:
            header_xy.append(cell)

x_axis =1
y_axis=0
header_num=len(set(header_list))





#関数を入力
for index, x in enumerate(header_xy):
        clock = f'{Sheet_one.cell(x.row+3, x.column-1).coordinate}:{Sheet_one.cell(max_row, x.column-1).coordinate}'#時刻
        ref = f'{Sheet_one.cell(x.row+3, x.column).coordinate}:{Sheet_one.cell(max_row, x.column).coordinate}'#バーストサイズ     
        ref2 = f'{summary_ws.cell(x.row+3, x.column-1).coordinate}:{summary_ws.cell(max_row, x.column-1).coordinate}'
        ref3 = f'{summary_ws.cell(x.row+3, x.column).coordinate}:{summary_ws.cell(max_row, x.column).coordinate}'#IBI
        
        clock2= f'{lick_rate_ws.cell(x.row+3, x.column-1).coordinate}:{lick_rate_ws.cell(max_row, x.column-1).coordinate}'#時刻
        ref4=f'{lick_rate_ws.cell(x.row+3, x.column).coordinate}:{lick_rate_ws.cell(max_row, x.column).coordinate}'

        if len(header_list) > 0:  
            header_list.pop(0)
        if date_flag:
            summary_ws.cell(1+x_axis,3+y_axis).value= '=IF(A3>A2,AVERAGEIFS(Sheet!'+str(ref)+',Sheet!'+str(clock)+',">="&A2,Sheet!'+str(clock)+',"<="&A3),(SUMIFS(Sheet!'+str(ref)+',Sheet!'+str(clock)+',">="&A2,Sheet!'+str(clock)+',"<=23:59:59")+SUMIFS(Sheet!'+str(ref)+',Sheet!'+str(clock)+',">=0:00",Sheet!'+str(clock)+',"<="&A3))/(COUNTIFS(Sheet!'+str(clock)+',">="&A2,Sheet!'+str(clock)+',"<=23:59:59")+COUNTIFS(Sheet!'+str(clock)+',">=0:00",Sheet!'+str(clock)+',"<="&A3)))'
            summary_ws.cell(1+x_axis,4+header_num+y_axis).value= '=IF(A3>A2,COUNTIFS(Sheet!'+str(clock)+',">="&A2,Sheet!'+str(clock)+',"<="&A3),COUNTIFS(Sheet!'+str(clock)+',">="&A2,Sheet!'+str(clock)+',"<=23:59:59")+COUNTIFS(Sheet!'+str(clock)+',">=0:00",Sheet!'+str(clock)+',"<="&A3))'
            summary_ws.cell(1+x_axis,5+header_num*2+y_axis).value= '=IF(A3>A2,SUMIFS(Sheet!'+str(ref)+',Sheet!'+str(clock)+',">="&A2,Sheet!'+str(clock)+',"<="&A3),SUMIFS(Sheet!'+str(ref)+',Sheet!'+str(clock)+',">="&A2,Sheet!'+str(clock)+',"<=23:59:59")+SUMIFS(Sheet!'+str(ref)+',Sheet!'+str(clock)+',">=0:00",Sheet!'+str(clock)+',"<="&A3))'
            summary_ws.cell(1+x_axis,6+header_num*3+y_axis).value= '=IF(A3>A2,AVERAGEIFS(TimeAndIBI!'+str(ref2)+',Sheet!'+str(clock)+',">="&A2,Sheet!'+str(clock)+',"<="&A3),(SUMIFS(TimeAndIBI!'+str(ref2)+',Sheet!'+str(clock)+',">="&A2,Sheet!'+str(clock)+',"<=23:59:59")+SUMIFS(TimeAndIBI!'+str(ref2)+',Sheet!'+str(clock)+',">=0:00",Sheet!'+str(clock)+',"<="&A3))/(COUNTIFS(Sheet!'+str(clock)+',">="&A2,Sheet!'+str(clock)+',"<=23:59:59")+COUNTIFS(Sheet!'+str(clock)+',">=0:00",Sheet!'+str(clock)+',"<="&A3)))'
            summary_ws.cell(1+x_axis,7+header_num*4+y_axis).value= '=IF(A3>A2,AVERAGEIFS(TimeAndIBI!'+str(ref3)+',Sheet!'+str(clock)+',">="&A2,Sheet!'+str(clock)+',"<="&A3),(SUMIFS(TimeAndIBI!'+str(ref3)+',Sheet!'+str(clock)+',">="&A2,Sheet!'+str(clock)+',"<=23:59:59")+SUMIFS(TimeAndIBI!'+str(ref3)+',Sheet!'+str(clock)+',">=0:00",Sheet!'+str(clock)+',"<="&A3))/(COUNTIFS(Sheet!'+str(clock)+',">="&A2,Sheet!'+str(clock)+',"<=23:59:59")+COUNTIFS(Sheet!'+str(clock)+',">=0:00",Sheet!'+str(clock)+',"<="&A3)))'
            summary_ws.cell(1+x_axis,8+header_num*5+y_axis).value= '=IF(A3>A2,AVERAGEIFS(lick_rate!'+str(ref4)+',lick_rate!'+str(clock2)+',">="&A2,lick_rate!'+str(clock2)+',"<="&A3),(SUMIFS(lick_rate!'+str(ref4)+',lick_rate!'+str(clock2)+',">="&A2,lick_rate!'+str(clock2)+',"<=23:59:59")+SUMIFS(lick_rate!'+str(ref4)+',lick_rate!'+str(clock2)+',">=0:00",lick_rate!'+str(clock2)+',"<="&A3))/(COUNTIFS(lick_rate!'+str(clock2)+',">="&A2,lick_rate!'+str(clock2)+',"<=23:59:59")+COUNTIFS(lick_rate!'+str(clock2)+',">=0:00",lick_rate!'+str(clock2)+',"<="&A3)))'
        else:
            clock_slot = f'Sheet!{clock},">="&A2,Sheet!{clock},"<="&A3'
            clock_slot2 = f'lick_rate!{clock2},">="&A2,lick_rate!{clock2},"<="&A3'
            summary_ws.cell(1+x_axis,3+y_axis).value = f'=AVERAGEIFS(Sheet!{ref},{clock_slot})'
            summary_ws.cell(1+x_axis,4+header_num+y_axis).value = f'=COUNTIFS({clock_slot})'
            summary_ws.cell(1+x_axis,5+header_num*2+y_axis).value = f'=SUMIFS(Sheet!{ref},{clock_slot})'
            summary_ws.cell(1+x_axis,6+header_num*3+y_axis).value = f'=AVERAGEIFS(TimeAndIBI!{ref2},{clock_slot})'
            summary_ws.cell(1+x_axis,7+header_num*4+y_axis).value = f'=AVERAGEIFS(TimeAndIBI!{ref3},{clock_slot})'
            summary_ws.cell(1+x_axis,8+header_num*5+y_axis).value = f'=AVERAGEIFS(lick_rate!{ref4},{clock_slot2})'

        x_axis +=1

        if str(x.value) not in header_list:
            y_axis += 1
            x_axis =1
 
       

excel.move_sheet('Summary', offset=-3)    
excel.save(filesave_path + "/" + save_file_name + ".xlsx")
excel.close()
wb2.close()
wb.close()

#関数参照のエラーを回避するために更新
filesave_path = r"/Users/shun/Desktop/Python/summary"
xb=xw.Book(filesave_path + "/" + save_file_name + ".xlsx")
xs=xb.sheets[1]
xs2=xb.sheets[3]
xs.range((1,1), (Sheet_one.max_row,Sheet_one.max_column)).value=xs.range((1,1), (Sheet_one.max_row,Sheet_one.max_column)).value
xs2.range((1,1), (lick_rate_ws.max_row,lick_rate_ws.max_column)).value=xs2.range((1,1), (lick_rate_ws.max_row,lick_rate_ws.max_column)).value

xb.save()


print('終了しました')

OSError: [WinError -2147467259] エラーを特定できません

In [51]:
'''
ここのセルではShift+Enterを押さない
URLのlocalhostが8888でない場合、再起動する必要がある
また、OSError: [WinError -2147467259] エラーを特定できません　というメッセージは無視して、ファイルを確認してください。
'''


'\nここのセルではShift+Enterを押さない\nURLのlocalhostが8888でない場合、再起動する必要がある\nまた、OSError: [WinError -2147467259] エラーを特定できません\u3000というメッセージは無視して、ファイルを確認してください。\n'

In [7]:
filesave_path + "/" + save_file_name + ".xlsx"

'/Users/shun/Desktop/Python/summary/Summary.xlsx'